# XGBoost CV Experiments — Early Stopping & Fold Averaging

Compares three training strategies against the **feature_eng joint best** tuned XGBoost baseline.

Reuses pipeline from `feature_eng.ipynb` / `models copy.ipynb`:
- `engineer_all_features()`
- Best feature columns + hyperparameters from `feature_eng_joint_best.json`
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`

**Primary metric:** log loss · **Secondary:** ROC-AUC

Competition `test.csv` is used only for fold-averaged test predictions (experiments 2–3). It is **never** used for early stopping or model selection.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_copy_utils import RANDOM_STATE, CV_FOLDS, build_tuned_xgb, evaluate_xgb_cv, load_data
from feature_eng_lib import engineer_all_features, load_feature_eng_best

EARLY_STOPPING_ROUNDS = 50
EARLY_STOPPING_N_ESTIMATORS_CAP = 2000

c:\Users\Nick\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data and tuned configuration

In [2]:
best_config = load_feature_eng_best()
FEATURE_COLS = best_config["best_feature_cols"]
XGB_PARAMS = best_config["best_xgb_params"]

train_df = engineer_all_features(load_data("train.csv"))
test_df = engineer_all_features(load_data("test.csv"))
y = train_df["default"]

X = train_df[FEATURE_COLS]
X_test = test_df[FEATURE_COLS]

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(cv.split(X, y))

print(f"Features: {len(FEATURE_COLS)}")
print(f"Train rows: {len(X):,} | Test rows: {len(X_test):,}")
print(f"Tuned n_estimators: {XGB_PARAMS['n_estimators']}")
print(f"Saved best val log loss: {best_config['best_val_log_loss_mean']:.6f}")

Features: 54
Train rows: 24,000 | Test rows: 6,000
Tuned n_estimators: 450
Saved best val log loss: 0.423229


## 3. Shared helpers

In [3]:
def aggregate_cv_metrics(
    y_true: pd.Series,
    oof_probs: np.ndarray,
    folds,
    fold_rows: list[dict] | None = None,
) -> dict:
    y_values = np.asarray(y_true)
    fold_val_ll = [
        log_loss(y_values[val_idx], oof_probs[val_idx]) for _, val_idx in folds
    ]
    fold_val_auc = [
        roc_auc_score(y_values[val_idx], oof_probs[val_idx]) for _, val_idx in folds
    ]

    out = {
        "val_log_loss_mean": float(np.mean(fold_val_ll)),
        "val_log_loss_std": float(np.std(fold_val_ll)),
        "val_roc_auc_mean": float(np.mean(fold_val_auc)),
        "val_roc_auc_std": float(np.std(fold_val_auc)),
    }

    if fold_rows:
        out["train_log_loss_mean"] = float(np.mean([r["train_log_loss"] for r in fold_rows]))
        out["train_roc_auc_mean"] = float(np.mean([r["train_roc_auc"] for r in fold_rows]))
        out["train_val_log_loss_gap"] = (
            out["train_log_loss_mean"] - out["val_log_loss_mean"]
        )
        if "best_iteration" in fold_rows[0]:
            out["mean_best_iteration"] = float(
                np.mean([r["best_iteration"] for r in fold_rows])
            )
            out["std_best_iteration"] = float(
                np.std([r["best_iteration"] for r in fold_rows])
            )
    else:
        cv_scores = evaluate_xgb_cv(build_tuned_xgb(XGB_PARAMS), X, y, cv=cv)
        out["train_log_loss_mean"] = cv_scores["train_log_loss_mean"]
        out["train_roc_auc_mean"] = cv_scores["train_roc_auc_mean"]
        out["train_val_log_loss_gap"] = (
            out["train_log_loss_mean"] - out["val_log_loss_mean"]
        )

    return out


def make_early_stopping_params(base_params: dict) -> dict:
    params = {k: v for k, v in base_params.items() if k != "n_estimators"}
    params["n_estimators"] = EARLY_STOPPING_N_ESTIMATORS_CAP
    return params


def fit_predict_fold(
    X_tr,
    y_tr,
    X_va,
    y_va,
    params: dict,
    early_stopping: bool = False,
):
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
        **(
            {"early_stopping_rounds": EARLY_STOPPING_ROUNDS}
            if early_stopping
            else {}
        ),
    )

    if early_stopping:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        best_iter = int(model.best_iteration)
        val_prob = model.predict_proba(X_va, iteration_range=(0, best_iter + 1))[:, 1]
        train_prob = model.predict_proba(X_tr, iteration_range=(0, best_iter + 1))[:, 1]
        test_prob = model.predict_proba(X_test, iteration_range=(0, best_iter + 1))[:, 1]
        best_iteration = best_iter + 1
    else:
        model.fit(X_tr, y_tr, verbose=False)
        val_prob = model.predict_proba(X_va)[:, 1]
        train_prob = model.predict_proba(X_tr)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]
        best_iteration = params.get("n_estimators")

    return {
        "model": model,
        "val_prob": val_prob,
        "train_prob": train_prob,
        "test_prob": test_prob,
        "best_iteration": best_iteration,
        "train_log_loss": float(log_loss(y_tr, train_prob)),
        "val_log_loss": float(log_loss(y_va, val_prob)),
        "train_roc_auc": float(roc_auc_score(y_tr, train_prob)),
        "val_roc_auc": float(roc_auc_score(y_va, val_prob)),
    }

## 4. Baseline — tuned fixed-tree XGBoost (same folds)

Single model config with fixed `n_estimators` from feature_eng best; standard sklearn CV.

In [4]:
baseline_model = build_tuned_xgb(XGB_PARAMS)
baseline_oof = cross_val_predict(
    baseline_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

baseline_cv = evaluate_xgb_cv(baseline_model, X, y, cv=cv)
baseline_metrics = aggregate_cv_metrics(y, baseline_oof, FOLDS)
baseline_metrics["experiment"] = "baseline (tuned fixed trees)"
baseline_metrics["mean_best_iteration"] = float(XGB_PARAMS["n_estimators"])
baseline_metrics["std_best_iteration"] = 0.0

print("Baseline fold-equivalent CV:")
for k, v in baseline_metrics.items():
    if k != "experiment":
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

Baseline fold-equivalent CV:
  val_log_loss_mean: 0.423229
  val_log_loss_std: 0.006669
  val_roc_auc_mean: 0.789595
  val_roc_auc_std: 0.010219
  train_log_loss_mean: 0.391379
  train_roc_auc_mean: 0.835216
  train_val_log_loss_gap: -0.031850
  mean_best_iteration: 450.000000
  std_best_iteration: 0.000000


## 5. Experiment 1 — Early stopping only

Large `n_estimators` cap with fold-specific validation early stopping. Full OOF probabilities.

In [5]:
es_params = make_early_stopping_params(XGB_PARAMS)
es_oof = np.zeros(len(y))
es_fold_rows = []

for fold_idx, (train_idx, val_idx) in enumerate(FOLDS, start=1):
    result = fit_predict_fold(
        X.iloc[train_idx],
        y.iloc[train_idx],
        X.iloc[val_idx],
        y.iloc[val_idx],
        es_params,
        early_stopping=True,
    )
    es_oof[val_idx] = result["val_prob"]
    es_fold_rows.append(
        {
            "fold": fold_idx,
            "best_iteration": result["best_iteration"],
            "train_log_loss": result["train_log_loss"],
            "val_log_loss": result["val_log_loss"],
            "train_roc_auc": result["train_roc_auc"],
            "val_roc_auc": result["val_roc_auc"],
        }
    )

es_fold_report = pd.DataFrame(es_fold_rows)
es_metrics = aggregate_cv_metrics(y, es_oof, FOLDS, es_fold_rows)
es_metrics["experiment"] = "early stopping only"

print("Experiment 1 — per-fold results:")
print(es_fold_report.to_string(index=False))
print("\nAggregated:")
print(pd.Series({k: v for k, v in es_metrics.items() if k != "experiment"}))

Experiment 1 — per-fold results:
 fold  best_iteration  train_log_loss  val_log_loss  train_roc_auc  val_roc_auc
    1             380        0.392215      0.432272       0.834061     0.775486
    2             392        0.394257      0.424945       0.832026     0.786712
    3             418        0.396415      0.412484       0.829015     0.806466
    4             419        0.392934      0.422465       0.833408     0.788814
    5             383        0.394512      0.426397       0.830945     0.788251

Aggregated:
val_log_loss_mean           0.423713
val_log_loss_std            0.006475
val_roc_auc_mean            0.789146
val_roc_auc_std             0.009934
train_log_loss_mean         0.394067
train_roc_auc_mean          0.831891
train_val_log_loss_gap     -0.029646
mean_best_iteration       398.400000
std_best_iteration         16.883128
dtype: float64


## 6. Experiment 2 — Fold-model averaging only

Fixed tuned trees; one model per fold; average fold predictions on test.

In [6]:
avg_oof = np.zeros(len(y))
avg_test_fold_probs = []
avg_fold_rows = []

for fold_idx, (train_idx, val_idx) in enumerate(FOLDS, start=1):
    result = fit_predict_fold(
        X.iloc[train_idx],
        y.iloc[train_idx],
        X.iloc[val_idx],
        y.iloc[val_idx],
        XGB_PARAMS,
        early_stopping=False,
    )
    avg_oof[val_idx] = result["val_prob"]
    avg_test_fold_probs.append(result["test_prob"])
    avg_fold_rows.append(
        {
            "fold": fold_idx,
            "best_iteration": result["best_iteration"],
            "train_log_loss": result["train_log_loss"],
            "val_log_loss": result["val_log_loss"],
            "train_roc_auc": result["train_roc_auc"],
            "val_roc_auc": result["val_roc_auc"],
        }
    )

avg_test_fold_probs = np.vstack(avg_test_fold_probs)
avg_test_probs = avg_test_fold_probs.mean(axis=0)
avg_test_std = avg_test_fold_probs.std(axis=0)

avg_fold_report = pd.DataFrame(avg_fold_rows)
avg_metrics = aggregate_cv_metrics(y, avg_oof, FOLDS, avg_fold_rows)
avg_metrics["experiment"] = "fold averaging only"
avg_metrics["test_prob_std_mean"] = float(avg_test_std.mean())
avg_metrics["test_prob_std_median"] = float(np.median(avg_test_std))
avg_metrics["test_prob_std_max"] = float(avg_test_std.max())

print("Experiment 2 — per-fold OOF results:")
print(avg_fold_report.to_string(index=False))
print("\nTest prediction dispersion across folds:")
print(f"  mean row std:   {avg_metrics['test_prob_std_mean']:.6f}")
print(f"  median row std: {avg_metrics['test_prob_std_median']:.6f}")
print(f"  max row std:    {avg_metrics['test_prob_std_max']:.6f}")

Experiment 2 — per-fold OOF results:
 fold  best_iteration  train_log_loss  val_log_loss  train_roc_auc  val_roc_auc
    1             450        0.388262      0.432601       0.838964     0.775041
    2             450        0.390692      0.425353       0.836758     0.785871
    3             450        0.394577      0.412565       0.831483     0.806324
    4             450        0.391059      0.422641       0.835844     0.788628
    5             450        0.390386      0.426531       0.836350     0.788055

Test prediction dispersion across folds:
  mean row std:   0.013229
  median row std: 0.011066
  max row std:    0.069105


## 7. Experiment 3 — Early stopping + fold averaging

In [7]:
es_avg_oof = np.zeros(len(y))
es_avg_test_fold_probs = []
es_avg_fold_rows = []

for fold_idx, (train_idx, val_idx) in enumerate(FOLDS, start=1):
    result = fit_predict_fold(
        X.iloc[train_idx],
        y.iloc[train_idx],
        X.iloc[val_idx],
        y.iloc[val_idx],
        es_params,
        early_stopping=True,
    )
    es_avg_oof[val_idx] = result["val_prob"]
    es_avg_test_fold_probs.append(result["test_prob"])
    es_avg_fold_rows.append(
        {
            "fold": fold_idx,
            "best_iteration": result["best_iteration"],
            "train_log_loss": result["train_log_loss"],
            "val_log_loss": result["val_log_loss"],
            "train_roc_auc": result["train_roc_auc"],
            "val_roc_auc": result["val_roc_auc"],
        }
    )

es_avg_test_fold_probs = np.vstack(es_avg_test_fold_probs)
es_avg_test_probs = es_avg_test_fold_probs.mean(axis=0)
es_avg_test_std = es_avg_test_fold_probs.std(axis=0)

es_avg_fold_report = pd.DataFrame(es_avg_fold_rows)
es_avg_metrics = aggregate_cv_metrics(y, es_avg_oof, FOLDS, es_avg_fold_rows)
es_avg_metrics["experiment"] = "early stopping + fold averaging"
es_avg_metrics["test_prob_std_mean"] = float(es_avg_test_std.mean())
es_avg_metrics["test_prob_std_median"] = float(np.median(es_avg_test_std))
es_avg_metrics["test_prob_std_max"] = float(es_avg_test_std.max())

print("Experiment 3 — per-fold results:")
print(es_avg_fold_report.to_string(index=False))
print("\nTest prediction dispersion across folds:")
print(f"  mean row std:   {es_avg_metrics['test_prob_std_mean']:.6f}")
print(f"  median row std: {es_avg_metrics['test_prob_std_median']:.6f}")
print(f"  max row std:    {es_avg_metrics['test_prob_std_max']:.6f}")

Experiment 3 — per-fold results:
 fold  best_iteration  train_log_loss  val_log_loss  train_roc_auc  val_roc_auc
    1             380        0.392215      0.432272       0.834061     0.775486
    2             392        0.394257      0.424945       0.832026     0.786712
    3             418        0.396415      0.412484       0.829015     0.806466
    4             419        0.392934      0.422465       0.833408     0.788814
    5             383        0.394512      0.426397       0.830945     0.788251

Test prediction dispersion across folds:
  mean row std:   0.012521
  median row std: 0.010408
  max row std:    0.062842


## 8. Comparison table

In [8]:
comparison_cols = [
    "experiment",
    "train_log_loss_mean",
    "val_log_loss_mean",
    "val_log_loss_std",
    "train_val_log_loss_gap",
    "val_roc_auc_mean",
    "val_roc_auc_std",
    "train_roc_auc_mean",
    "mean_best_iteration",
    "std_best_iteration",
    "test_prob_std_mean",
]

comparison = pd.DataFrame(
    [
        baseline_metrics,
        es_metrics,
        avg_metrics,
        es_avg_metrics,
    ]
)[comparison_cols]

comparison = comparison.sort_values("val_log_loss_mean")

print("Comparison (lower val log loss is better):")
print(comparison.to_string(index=False))

comparison

Comparison (lower val log loss is better):
                     experiment  train_log_loss_mean  val_log_loss_mean  val_log_loss_std  train_val_log_loss_gap  val_roc_auc_mean  val_roc_auc_std  train_roc_auc_mean  mean_best_iteration  std_best_iteration  test_prob_std_mean
   baseline (tuned fixed trees)             0.391379           0.423229          0.006669               -0.031850          0.789595         0.010219            0.835216                450.0            0.000000                 NaN
            early stopping only             0.394067           0.423713          0.006475               -0.029646          0.789146         0.009934            0.831891                398.4           16.883128                 NaN
early stopping + fold averaging             0.394067           0.423713          0.006475               -0.029646          0.789146         0.009934            0.831891                398.4           16.883128            0.012521
            fold averaging only      

,experiment,train_log_loss_mean,val_log_loss_mean,val_log_loss_std,train_val_log_loss_gap,val_roc_auc_mean,val_roc_auc_std,train_roc_auc_mean,mean_best_iteration,std_best_iteration,test_prob_std_mean
0,baseline (tuned fixed trees),0.391379,0.423229,0.006669,-0.031850,0.789595,0.010219,0.835216,450.0,0.000000,NaN
1,early stopping only,0.394067,0.423713,0.006475,-0.029646,0.789146,0.009934,0.831891,398.4,16.883128,NaN
3,early stopping + fold averaging,0.394067,0.423713,0.006475,-0.029646,0.789146,0.009934,0.831891,398.4,16.883128,0.012521
2,fold averaging only,0.390995,0.423938,0.006554,-0.032943,0.788784,0.010055,0.835880,450.0,0.000000,0.013229


## 9. Optional — save fold-averaged test submissions

In [9]:
pd.DataFrame(
    {
        "client_id": test_df["client_id"],
        "default_probability": avg_test_probs,
    }
).to_csv("submission_fold_avg.csv", index=False)

pd.DataFrame(
    {
        "client_id": test_df["client_id"],
        "default_probability": es_avg_test_probs,
    }
).to_csv("submission_es_fold_avg.csv", index=False)

print("Saved submission_fold_avg.csv and submission_es_fold_avg.csv")

Saved submission_fold_avg.csv and submission_es_fold_avg.csv
